In [27]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel,Field
from typing import TypedDict, Literal
from dotenv import load_dotenv
import os
load_dotenv()

True

In [28]:
model=ChatGoogleGenerativeAI(
model='gemini-3.5-flash-lite',google_api_key=os.getenv("GEMINI_API_KEY_3"))

In [29]:
class Sentiment(BaseModel):
    sentiment:Literal['posetive','negative'] = Field(description="Sentiment of the review")

In [30]:
class Diagnosis(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [31]:
structured_model=model.with_structured_output(Sentiment)
structured_model2=model.with_structured_output(Diagnosis)

In [32]:
prompt="What is the sentiment of the following review: The software is very bad"
res=structured_model.invoke(prompt)
res.sentiment

'negative'

In [33]:
type(res.sentiment)

str

# START

In [41]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal['posetive','negative']
    diagnosis:dict
    response:str

In [42]:
def findSentiment(state:ReviewState):
    prompt=f'For the following review find out the sentiment \n {state['review']}'
    sentiment=structured_model.invoke(prompt).sentiment
    return {'sentiment':sentiment}

In [43]:
def checkCondition(state:ReviewState) -> Literal['posetiveResponse','runDiagnosis']:
    if state['sentiment']=='posetive':
        return 'posetiveResponse'
    else:
        return 'runDiagnosis'

In [44]:
def posetiveResponse(state:ReviewState):
    prompt=f"Write a warm thank you message in response to this review: \n\n {state['review']} \n Also kindly ask the user to leave a feedback in the website"
    res=model.invoke(prompt).content[0]['text']
    return {'response':res}

In [ ]:
def runDiagnosis(state:ReviewState):
    prompt=f"Diagnose this negative review: \n\n{state['review']}\n Return issue_type,tone and urgency."
    res=structured_model2.invoke(prompt)
    return {'diagnosis':res.model_dump()} #model_dump changes to dictionary

In [46]:
def negativeResponse(state: ReviewState):
    diagnosis=state['diagnosis']
    prompt = f"""You are a support assistant.
    The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
    Write an empathetic, helpful resolution message.
    """
    res=model.invoke(prompt).content[0]['text']
    return {'response':res}

In [47]:
graph=StateGraph(ReviewState)

graph.add_node('findSentiment',findSentiment)
graph.add_node('posetiveResponse',posetiveResponse)
graph.add_node('runDiagnosis',runDiagnosis)
graph.add_node('negativeResponse',negativeResponse)

graph.add_edge(START,'findSentiment')
graph.add_conditional_edges('findSentiment',checkCondition)
graph.add_edge('posetiveResponse',END)
graph.add_edge('runDiagnosis','negativeResponse')

graph.add_edge('negativeResponse',END)

workflow=graph.compile()

In [48]:
initial_state={
    'review': "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."}

workflow.invoke(initial_state)

{'review': 'I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Bug', 'tone': 'frustrated', 'urgency': 'high'},
 'response': 'I completely understand how frustrating it is to run into a bug, especially when you need things to work smoothly and urgently. I am very sorry for the inconvenience this has caused you. \n\nI’ve taken a close look at this issue and have escalated it to our engineering team with high priority. We are actively working on a fix right now, and I will monitor the progress personally to ensure it gets resolved as quickly as possible. \n\nI will follow up with you as soon as an update is available or the patch is applied. Thank you for your patience and for bringing this to our attention.'}